49 points

In [ ]:
import pandas as pd
from PIL import Image
from tqdm import tqdm
import numpy as np
import cv2

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, PowerTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.pipeline import make_pipeline

import torch
from transformers import CLIPProcessor, CLIPModel, AutoImageProcessor, AutoModel

In [ ]:
root_path = "/home/stefan/Downloads/dataset-b22bcb58-37e6-4196-adec-3bcb60881e46"
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

In [ ]:
dino_processor = AutoImageProcessor.from_pretrained(
    "facebook/dinov3-vitb16-pretrain-lvd1689m"
)
dino_model = AutoModel.from_pretrained("facebook/dinov3-vitb16-pretrain-lvd1689m").to(
    device
)

clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

In [ ]:
train_df_init = pd.read_csv(f"{root_path}/train.csv")
le = LabelEncoder()
le.fit(train_df_init["Type"])
all_types = le.classes_

In [ ]:
templates = [
    "a pokemon of type {}",
    "a {} type pokemon",
    "a drawing of a {} pokemon",
    "illustration of a {} pokemon",
]

print("Pre-computing Text Embeddings for Zero-Shot...")
text_features = []
with torch.no_grad():
    for t in all_types:
        # Create multiple prompts for this type and average them
        prompts = [temp.format(t) for temp in templates]
        inputs = clip_processor(text=prompts, return_tensors="pt", padding=True).to(
            device
        )
        text_emb = clip_model.get_text_features(**inputs)
        # Average the prompts to get a robust 'Type Prototype'
        text_emb = text_emb.mean(dim=0, keepdim=True)
        text_emb = text_emb / text_emb.norm(p=2, dim=-1, keepdim=True)
        text_features.append(text_emb)

# Shape: (18, 768) - These are the "Anchors" for our 18 classes
type_text_embeds = torch.vstack(text_features)

In [ ]:
def load_image_clean(path):
    img = Image.open(path).convert("RGBA")
    white_bg = Image.new("RGBA", img.size, "WHITE")
    white_bg.paste(img, (0, 0), img)
    return white_bg.convert("RGB")


def get_color_stats(path):
    # Simple color stats 
    img = cv2.imread(path)
    if img is None:
        return np.zeros(6)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = np.all(img > 250, axis=2)
    img = img[~mask]
    if len(img) == 0:
        return np.zeros(6)
    return np.concatenate([np.mean(img, axis=0), np.std(img, axis=0)])


def extract_ultimate_features(df):

    dino_feats = []
    clip_image_feats = []
    clip_probs = []
    color_feats = []

    print(f"Extracting features for {len(df)} images...")

    for _, row in tqdm(df.iterrows(), total=len(df)):
        path = f"{root_path}/{row['ImagePath']}"
        pil_img = load_image_clean(path)

        # A. DINO
        inputs = dino_processor(images=pil_img, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = dino_model(**inputs)
            dino_feats.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())

        # B. CLIP Image Embedding
        inputs = clip_processor(images=pil_img, return_tensors="pt").to(device)
        with torch.no_grad():
            img_emb = clip_model.get_image_features(**inputs)
            img_emb = img_emb / img_emb.norm(p=2, dim=-1, keepdim=True)

            # C. CLIP Zero-Shot Scores (Dot product with Text Embeddings)
            # This calculates similarity to "Grass", "Fire", "Water"...
            similarity = (100.0 * img_emb @ type_text_embeds.T).softmax(dim=-1)

            clip_image_feats.append(img_emb.cpu().numpy())
            clip_probs.append(similarity.cpu().numpy())

        # D. Color
        color_feats.append(get_color_stats(path))

    return np.hstack(
        [
            np.vstack(dino_feats),  # 1024 dims
            np.vstack(clip_image_feats),  # 768 dims
            np.vstack(clip_probs) * 10,  # 18 dims (scaled up to emphasize importance)
            np.vstack(color_feats),  # 6 dims
        ]
    )

In [ ]:
train_df = pd.read_csv(f"{root_path}/train.csv")
test_df = pd.read_csv(f"{root_path}/test.csv")

# Only fit LE on train, but ensuring we have the same classes as initialized
y_train = le.transform(train_df["Type"])

X_train = extract_ultimate_features(train_df)
X_test = extract_ultimate_features(test_df)

In [ ]:
# 5. The Ensemble Strategy
# We use a Voting Classifier to combine 3 mathematically different models.
# This reduces the variance (instability) caused by small data.

# 1. Logistic Regression (Robust Baseline)
clf1 = make_pipeline(
    PowerTransformer(),
    LogisticRegression(
        C=0.1, max_iter=3000, class_weight="balanced", solver="liblinear"
    ),
)

# 2. Support Vector Machine (Non-linear RBF Kernel)
clf2 = make_pipeline(
    StandardScaler(),
    SVC(C=1.0, kernel="rbf", probability=True, class_weight="balanced"),
)

# 3. MLP
# Tiny network to prevent overfitting
clf3 = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(128,), alpha=0.1, max_iter=1000, random_state=seed
    ),
)

# Ensemble
eclf = VotingClassifier(
    estimators=[("lr", clf1), ("svm", clf2), ("mlp", clf3)],
    voting="soft",  # Average the probabilities
    weights=[
        1,
        2,
        1,
    ],  # Give SVM slightly more weight as it's usually best for small-data high-dim
)

In [ ]:
scores = cross_val_score(eclf, X_train, y_train, cv=5, scoring="accuracy")
print(
    f"Cross-Validation Accuracy: {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f})"
)

In [ ]:
# 7. Final Training & Submit
print("Retraining on full data...")
eclf.fit(X_train, y_train)

test_preds = eclf.predict(X_test)
test_labels = le.inverse_transform(test_preds)

submission = pd.DataFrame({"SampleID": test_df["SampleID"], "Type": test_labels})
submission.to_csv(f"{root_path}/submission.csv", index=False)
print("Submission saved.")